# DeepSeek-OCR accuracy test (HarfScan FYP)

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells from top to bottom (Runtime → Run all).
3. When the upload button appears, select the **6 images** from `E:\New folder\deepseek_colab`.

The last cell prints the character error rate (CER) of DeepSeek-OCR next to our current pipeline (UTRNet + PaddleOCR). CER is measured the same way as in the project: letters only, harakat/punctuation/spaces ignored, Urdu/Arabic letter variants unified.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install transformers==4.46.3 tokenizers==0.20.3 einops addict easydict

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
name = 'deepseek-ai/DeepSeek-OCR'
tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
# T4 has no flash-attention 2 support -> plain (eager) attention.
model = AutoModel.from_pretrained(name, _attn_implementation='eager',
                                  trust_remote_code=True, use_safetensors=True)
model = model.eval().cuda().to(torch.bfloat16)
print('model loaded')

In [ ]:
from google.colab import files
uploaded = files.upload()   # select the 6 test images
print(sorted(uploaded))

In [ ]:
REFERENCE = {
 "real5.jpg": "فصل (۱۹) ۶۱ حروف عاملہ در فعل مضارع\nفصل (۱۹)\nحروف عاملہ در فعل مضارع\nیعنی مضارع پر عمل کرنے والے حروف، فعل مضارع میں جو حروف عمل کرتے ہیں ان کی\nدو قسمیں ہیں: ۱۔ ناصب ۲۔ جازم\nحروف ناصبہ یہ ہیں:\nأَنْ و لَنْ پھر كَيْ إِذَنْ ہیں یہ حروف ناصبہ\nکرتے ہیں منصوب مستقبل کو بے روئے وریا\nمثالیں\n۱۔ أَنْ جیسے: أُرِيدُ أَنْ تَقُومَ، أَنْ فعل کے ساتھ مصدر کے معنی میں ہوتا ہے، یعنی\nأُرِيدُ قِيَامَكَ، اس لیے اس کو أَنْ مصدریہ کہتے ہیں۔\n۲۔ لَنْ جیسے: لَنْ يَذْهَبَ عَمْرٌو، لَنْ تاکید نفی کے لیے آتا ہے۔\n۳۔ كَيْ جیسے: أَسْلَمْتُ كَيْ أَدْخُلَ الْجَنَّةَ۔\n۴۔ إِذَنْ جیسے: إِذَنْ أَشْكُرَكَ، اُس شخص کے جواب میں کہا جائے جو کہے: أَنَا\nأُعْطِيكَ دِينَارًا۔\nفائدہ: أَنْ چھ حروف کے بعد مقدر ہوتا ہے اور فعل مضارع کو نصب کرتا ہے:\n۱۔ حَتّٰى جیسے: مَرَرْتُ حَتّٰى أَدْخُلَ الْبَلَدَ۔\n۲۔ لام جَحْد جیسے: مَا كَانَ اللّٰهُ لِيُعَذِّبَهُمْ",
 "real1.png": "سبق نمبر ۸ کے الفاظ کے معانی\n\nسمع اس نے سنا شکر اس نے شکر کیا\nعمہ وہ بھٹکا علم اس نے جانا\nسال اس نے پوچھا الا مگر\nنصر اس نے مدد کی ایمان ایمان ایمان لانا ا م ن\nحزن وہ غمگین ہوا ما جو کچھ\nایدی میرا ہاتھ لما کیوں کس لیے\nثم پھر قطع اس نے کاٹا\nقول قول کہنا حذر وہ ڈرا\nہذا یہ فقہ اس نے سمجھا\nمن سے جہل اس نے جہالت کی\nعند نزدیک پاس سخر اس نے ہنسی اڑائی\nکرہ اس نے ناپسند کیا تلاوۃ تلاوت تلاوت کرنا ت ل و\nجمع اس نے جمع کیا لا نہیں\nخلد وہ ہمیشہ رہا\n",
 "real2.png": "سبق نمبر ۸ کے الفاظ کے معانی\n\nسمع اس نے سنا شکر اس نے شکر کیا\nعمہ وہ بھٹکا علم اس نے جانا\nسال اس نے پوچھا الا مگر\nنصر اس نے مدد کی ایمان ایمان ایمان لانا ا م ن\nحزن وہ غمگین ہوا ما جو کچھ\nایدی میرا ہاتھ لما کیوں کس لیے\nثم پھر قطع اس نے کاٹا\nقول قول کہنا حذر وہ ڈرا\nہذا یہ فقہ اس نے سمجھا\nمن سے جہل اس نے جہالت کی\nعند نزدیک پاس سخر اس نے ہنسی اڑائی\nکرہ اس نے ناپسند کیا تلاوۃ تلاوت تلاوت کرنا ت ل و\nجمع اس نے جمع کیا لا نہیں\nخلد وہ ہمیشہ رہا\n",
 "page_photo.jpg": "اردو ادب کی تاریخ\nاردو زبان برصغیر کی ایک اہم زبان ہے۔ اس زبان میں شاعری اور نثر کا بہت بڑا ذخیرہ موجود ہے۔ پرانی کتابوں کو محفوظ کرنا ہماری ذمہ داری ہے۔\nمیر تقی میر کی شاعری\nمرزا غالب کے خطوط\nعلامہ اقبال کا کلام\nبسم الله الرحمن الرحيم\nالحمد لله رب العالمين",
 "page_photo2.jpg": "اردو ادب کی تاریخ\nاردو زبان برصغیر کی ایک اہم زبان ہے۔ اس زبان میں شاعری اور نثر کا بہت بڑا ذخیرہ موجود ہے۔ پرانی کتابوں کو محفوظ کرنا ہماری ذمہ داری ہے۔\nمیر تقی میر کی شاعری\nمرزا غالب کے خطوط\nعلامہ اقبال کا کلام\nبسم الله الرحمن الرحيم\nالحمد لله رب العالمين",
 "page_clean.png": "اردو ادب کی تاریخ\nاردو زبان برصغیر کی ایک اہم زبان ہے۔ اس زبان میں شاعری اور نثر کا بہت بڑا ذخیرہ موجود ہے۔ پرانی کتابوں کو محفوظ کرنا ہماری ذمہ داری ہے۔\nمیر تقی میر کی شاعری\nمرزا غالب کے خطوط\nعلامہ اقبال کا کلام\nبسم الله الرحمن الرحيم\nالحمد لله رب العالمين"
}

BASELINE_CER = {
 "real5.jpg": 1.58,
 "real1.png": 2.13,
 "real2.png": 3.66,
 "page_photo.jpg": 1.0,
 "page_photo2.jpg": 0.5,
 "page_clean.png": 0.0
}

In [ ]:
import re, time, os
_DROP = re.compile(r"[\u064b-\u0670\u065f\u0640\u06d4\u060c\u061b\u061f.,:;()\[\]'\"\u2018\u2019\u201c\u201d\u2022|\-\u2013\u2014\s#*_>`$]")
_MAP = str.maketrans({'ي': 'ی', 'ى': 'ی', 'ئ': 'ی', 'ك': 'ک', 'ه': 'ہ', 'ة': 'ہ', 'ۃ': 'ہ', 'ۂ': 'ہ',
                      'ھ': 'ہ', 'أ': 'ا', 'إ': 'ا', 'آ': 'ا', 'ٱ': 'ا', 'ؤ': 'و', 'ء': '',
                      '٨': '۸', '8': '۸'})
def normalise(t):
    t = re.sub(r'<\|[^|]*\|>', '', t)          # DeepSeek control tokens
    t = re.sub(r'</?[a-z][^>]*>', '', t)          # html tags (tables)
    return _DROP.sub('', t).translate(_MAP)
def cer(pred, ref):
    a, b = normalise(pred), normalise(ref)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1] / max(1, len(b))

In [ ]:
PROMPT = '<image>\nFree OCR. '
results = {}
for img in REFERENCE:
    if img not in uploaded:
        print('missing', img); continue
    t = time.time()
    text = model.infer(tokenizer, prompt=PROMPT, image_file=img, output_path='out',
                       base_size=1024, image_size=640, crop_mode=True,
                       save_results=False, eval_mode=True)
    results[img] = (text, cer(text, REFERENCE[img]) * 100, time.time() - t)
    print(f'===== {img}  CER {results[img][1]:.2f}%  {results[img][2]:.0f}s')
    print(text)

In [ ]:
print(f"{'page':18} {'ours (UTRNet+Paddle)':>22} {'DeepSeek-OCR':>14} {'time':>6}")
for img, (text, c, sec) in results.items():
    print(f'{img:18} {BASELINE_CER[img]:21.2f}% {c:13.2f}% {sec:5.0f}s')
n = len(results)
print(f"{'AVERAGE':18} {sum(BASELINE_CER[i] for i in results)/n:21.2f}% "
      f"{sum(r[1] for r in results.values())/n:13.2f}%")